# MCFIX CKIP GPT2 Tiny Chinese: JED351-Matched Pipeline

This notebook extracts the same four Li-style metrics used by the current JED351 analysis, maps NR text-derived metrics to TSR by canonical text position, enforces the exact 10,089-row JED351 reference sample, runs the same five-seed CatBoost regressions and ablations, and downloads one ZIP.


## 1. Install dependencies


In [ ]:
%pip -q install "minicons==0.3.38" "transformers==5.13.1" "accelerate>=0.34" "catboost==1.2.10" "openpyxl>=3.1,<4" pandas numpy scipy tqdm


## 2. Extract CKIP GPT2-tiny-Chinese metrics from NR text


In [ ]:
from IPython.display import display


import copy
import importlib.metadata
import json
import math
import re
import unicodedata
import gc
import hashlib
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from minicons import scorer as minicons_scorer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, BertTokenizerFast

MODEL_ID = 'ckiplab/gpt2-tiny-chinese'
TOKENIZER_ID = 'bert-base-chinese'
MODEL_PREFIX = 'ckip_gpt2_tiny_'

INPUT_FILENAME = "trad_NR_first_column_aligned_original_content.xlsx"
CIFU_POS_FILENAME = "Cifu-v1_with_pycantonese_pos.csv"
BASE_MODELING_FILENAME = "MCFIX_NR_TSR_NEW_JED351_complete_cases.csv"
SHEET_NAME = "Original_Data"
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/MCFIX_CKIP_GPT2_tiny_JED351_matched/extraction"
)

SENTENCE_COL = "SENTENCE"
ORDER_COL = "n_in_sent"
PRIMARY_WORD_COL = "pure_token"
SURFACE_COL = "IA_LABEL"
PRIMARY_POS_COL = "upos*"
FALLBACK_POS_COL = "pos"

MAX_CONTEXT_TOKENS = 1024
INVALID_POS_FOR_POS_SURPRISAL = {"X"}
PRIMARY_POS_FALLBACK_TAGS = {"X", "DET"}
POS_ALIAS = {
    "N": "NOUN", "NN": "NOUN", "NNB": "NOUN", "NZ": "NOUN",
    "NR": "PROPN", "NS": "PROPN",
    "V": "VERB", "VV": "VERB", "VG": "VERB", "VN": "VERB",
    "A": "ADJ", "ADJ": "ADJ", "JJ": "ADJ",
    "D": "ADV", "ADV": "ADV", "RB": "ADV",
    "R": "PRON", "PRON": "PRON", "PRP": "PRON",
    "M": "NUM", "NUM": "NUM", "CD": "NUM",
    "P": "ADP", "ADP": "ADP", "IN": "ADP",
    "C": "CCONJ", "CC": "CCONJ", "CCONJ": "CCONJ",
    "U": "PART", "Y": "PART", "ZG": "PART", "PART": "PART",
    "DEC": "PART", "DEV": "PART", "SFN": "PART",
    "O": "INTJ", "E": "INTJ", "INTJ": "INTJ", "UH": "INTJ",
}

USE_DEVICE_MAP = False
EPS = 1e-12
CIFU_CANDIDATE_BATCH_SIZE = 512

# Stop before model loading if a normalized lexical target cannot be located
# unambiguously inside its normalized IA_LABEL surface form.
FAIL_ON_ALIGNMENT_ERROR = True

# Li's released scorer used raw causal-LM text rather than a chat template.
MINICONS_EOS_TOKEN = False
MINICONS_CHAT_MODE = False

# Set to a small integer, e.g. 50, for a quick smoke test.
RUN_LIMIT_ROWS = None
EXTRACTION_CHECKPOINT_VERSION = 'ckip-gpt2-tiny-jed351-matched-v1'
UNK_OFFSET_CONTENT_POLICY = 'score-nonzero-offset-unk-v2'

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ModuleNotFoundError:
    pass

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "sentence_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Model:", MODEL_ID)
print("Tokenizer:", TOKENIZER_ID)
print("Output directory:", OUTPUT_DIR)


input_path = Path("/content") / INPUT_FILENAME
cifu_pos_path = Path("/content") / CIFU_POS_FILENAME
base_modeling_path = Path("/content") / BASE_MODELING_FILENAME

if (
    not input_path.exists()
    or not cifu_pos_path.exists()
    or not base_modeling_path.exists()
):
    try:
        from google.colab import files

        print("Please upload:", INPUT_FILENAME, CIFU_POS_FILENAME, BASE_MODELING_FILENAME)
        files.upload()
    except ModuleNotFoundError:
        pass

if not input_path.exists():
    raise FileNotFoundError(
        f"Cannot find {INPUT_FILENAME}. Upload it to Colab /content or adjust input_path."
    )
if not cifu_pos_path.exists():
    raise FileNotFoundError(
        f"Cannot find {CIFU_POS_FILENAME}. Upload it to Colab /content or adjust cifu_pos_path."
    )
if not base_modeling_path.exists():
    raise FileNotFoundError(
        f"Cannot find {BASE_MODELING_FILENAME}. Upload it to Colab /content."
    )

print("Using input file:", input_path)
print("Using Cifu POS file:", cifu_pos_path)
print("Using JED351-matched modeling base:", base_modeling_path)

def sha256_input_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SOURCE_INPUT_SHA256 = sha256_input_file(input_path)
CIFU_FILE_SHA256 = sha256_input_file(cifu_pos_path)


full_df = pd.read_excel(input_path, sheet_name=SHEET_NAME)
full_df = full_df.sort_values(
    [SENTENCE_COL, ORDER_COL],
    kind="stable",
).reset_index(drop=True)

required_cols = [
    SENTENCE_COL,
    ORDER_COL,
    PRIMARY_WORD_COL,
    SURFACE_COL,
    PRIMARY_POS_COL,
    FALLBACK_POS_COL,
]
missing_cols = [column for column in required_cols if column not in full_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

if full_df[SENTENCE_COL].isna().any():
    raise ValueError(f"{SENTENCE_COL} contains missing values.")
if full_df[ORDER_COL].isna().any():
    raise ValueError(f"{ORDER_COL} contains missing values.")

df = full_df if RUN_LIMIT_ROWS is None else full_df.head(RUN_LIMIT_ROWS).copy()

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())


def clean_text_value(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def li_text_normalizer(text):
    """Exact replacements in Li et al.'s released utils.text_normalizer."""
    text = clean_text_value(text)
    text = text.replace("⋯", "⋯⋯")
    text = text.replace("…", "⋯⋯")
    text = text.replace(" ", "")
    text = text.replace("‘", "'")
    text = text.replace("’", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("B612", "b612")
    text = text.replace("—", "﹣")
    text = text.replace("噉", "咁")
    text = text.replace("\u200b", "")
    text = re.compile(r'^[。」]+(?=[\w\u4e00-\u9fa5])').sub("", text)
    return text


def remove_unicode_punctuation(text):
    text = clean_text_value(text)
    return "".join(
        character
        for character in text
        if not unicodedata.category(character).startswith("P")
    ).strip()


def has_linguistic_char(text):
    text = clean_text_value(text)
    return any(
        character.isalnum()
        or ("\u3400" <= character <= "\u9fff")
        or ("\uf900" <= character <= "\ufaff")
        for character in text
    )


def canonical_pos_tag(pos):
    pos = clean_text_value(pos)
    if not pos:
        return ""
    upper = pos.upper()
    return POS_ALIAS.get(upper, upper)


def valid_pos_for_pos_surprisal(pos):
    pos = canonical_pos_tag(pos)
    return bool(pos) and pos not in INVALID_POS_FOR_POS_SURPRISAL


def build_llm_pos(row):
    for column in (PRIMARY_POS_COL, FALLBACK_POS_COL):
        pos = canonical_pos_tag(row.get(column, ""))
        if column == PRIMARY_POS_COL and pos in PRIMARY_POS_FALLBACK_TAGS:
            continue
        if valid_pos_for_pos_surprisal(pos):
            return pos
    return ""


SCORABLE_ALIGNMENT = {
    "exact",
    "exact_with_lexical_prefix",
    "surface_missing_target_only",
}


def split_normalized_surface(row):
    """Return surface, lexical target, preceding material, trailing material, status.

    The lexical target must come from pure_token. Rows without pure_token are
    retained for reconstruction of the sentence context but receive no LLM
    metrics. A small number of MCFIX rows contain an extra lexical particle
    before pure_token inside the same IA_LABEL (e.g. IA_LABEL='㗎喇。' and
    pure_token='喇'). In those cases the extra prefix is retained as preceding
    context and the pure_token itself remains the scored lexical target.
    """
    surface = li_text_normalizer(row.get(SURFACE_COL, ""))
    target = li_text_normalizer(row.get(PRIMARY_WORD_COL, ""))

    # Match the 5,050 lexical rows used by the downstream Li-style regressions:
    # do not manufacture a target from IA_LABEL when pure_token is missing.
    if not target:
        return surface, "", surface, "", "missing_pure_token"

    if not surface:
        return target, target, "", "", "surface_missing_target_only"

    starts = []
    start = surface.find(target)
    while start >= 0:
        starts.append(start)
        start = surface.find(target, start + 1)

    exact_splits = []
    lexical_prefix_splits = []

    for start in starts:
        pre_context = surface[:start]
        post_context = surface[start + len(target):]

        # Ordinary case: everything outside pure_token is punctuation.
        if not has_linguistic_char(pre_context + post_context):
            exact_splits.append((pre_context, post_context))
            continue

        # MCFIX alignment exception: extra linguistic material occurs only
        # before pure_token, while everything after pure_token is punctuation.
        # Score pure_token conditional on that prefix and retain the full
        # normalized IA_LABEL in the context for subsequent words.
        if has_linguistic_char(pre_context) and not has_linguistic_char(post_context):
            lexical_prefix_splits.append((pre_context, post_context))

    if len(exact_splits) == 1:
        pre_context, post_context = exact_splits[0]
        return surface, target, pre_context, post_context, "exact"

    if len(exact_splits) > 1:
        return surface, target, "", "", "ambiguous_repeated_target"

    if len(lexical_prefix_splits) == 1:
        pre_context, post_context = lexical_prefix_splits[0]
        return (
            surface,
            target,
            pre_context,
            post_context,
            "exact_with_lexical_prefix",
        )

    if len(lexical_prefix_splits) > 1:
        return surface, target, "", "", "ambiguous_repeated_target"

    return surface, target, "", "", "target_not_aligned"


split_columns = [
    "_li_normalized_surface",
    "_llm_word_for_calc",
    "_llm_pre_context",
    "_llm_post_context",
    "_llm_alignment_status",
]
full_df[split_columns] = full_df.apply(
    lambda row: pd.Series(
        split_normalized_surface(row),
        index=split_columns,
    ),
    axis=1,
)
full_df["_llm_pos_for_calc"] = full_df.apply(build_llm_pos, axis=1)
full_df["_llm_metric_row_included"] = (
    full_df["_llm_alignment_status"].isin(SCORABLE_ALIGNMENT)
    & full_df["_llm_word_for_calc"].ne("")
    & full_df["_llm_word_for_calc"].map(has_linguistic_char)
)

df = full_df if RUN_LIMIT_ROWS is None else full_df.head(RUN_LIMIT_ROWS).copy()

acceptable_alignment = SCORABLE_ALIGNMENT | {
    "missing_pure_token",
    "punctuation_or_empty_row",
}
bad_alignment = ~df["_llm_alignment_status"].isin(acceptable_alignment)

print("Alignment status counts:")
print(df["_llm_alignment_status"].value_counts(dropna=False))
print(
    "Rows excluded because pure_token is missing:",
    int(df["_llm_alignment_status"].eq("missing_pure_token").sum()),
)
print(
    "Rows using a lexical prefix inside IA_LABEL as preceding context:",
    int(df["_llm_alignment_status"].eq("exact_with_lexical_prefix").sum()),
)
print(
    "Rows included in LLM metric extraction:",
    int(df["_llm_metric_row_included"].sum()),
)
print(
    "Rows where POS surprisal will be missing because no valid POS is available:",
    int(
        (
            df["_llm_metric_row_included"]
            & df["_llm_pos_for_calc"].eq("")
        ).sum()
    ),
)

if bad_alignment.any():
    print("Unresolved alignment rows:")
    display(
        df.loc[
            bad_alignment,
            [
                SENTENCE_COL,
                ORDER_COL,
                SURFACE_COL,
                PRIMARY_WORD_COL,
                "_li_normalized_surface",
                "_llm_word_for_calc",
                "_llm_alignment_status",
            ],
        ].head(50)
    )
    if FAIL_ON_ALIGNMENT_ERROR:
        raise ValueError(
            f"{int(bad_alignment.sum())} lexical rows could not be aligned "
            "unambiguously after Li-style normalization."
        )

display(
    df[
        [
            SENTENCE_COL,
            ORDER_COL,
            SURFACE_COL,
            PRIMARY_WORD_COL,
            "_li_normalized_surface",
            "_llm_pre_context",
            "_llm_word_for_calc",
            "_llm_post_context",
            "_llm_alignment_status",
            "_llm_metric_row_included",
            "_llm_pos_for_calc",
        ]
    ].head(30)
)


print("Loading tokenizer, model, and minicons scorer...")

tokenizer = BertTokenizerFast.from_pretrained(TOKENIZER_ID)
if not tokenizer.is_fast:
    raise RuntimeError(
        "A fast tokenizer is required for character-to-token offset alignment."
    )

if USE_DEVICE_MAP:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=False,
    )
    MINICONS_DEVICE = "auto"
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=False,
    )
    input_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(input_device)
    MINICONS_DEVICE = str(input_device)

model.eval()

# Reuse the already-loaded model; do not allocate a second 7B model.
lm_scorer = minicons_scorer.IncrementalLMScorer(
    model=model,
    device=MINICONS_DEVICE,
    tokenizer=tokenizer,
)
model = lm_scorer.model
tokenizer = lm_scorer.tokenizer
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device

automatic_special_count = tokenizer.num_special_tokens_to_add(pair=False)
MINICONS_BOS_TOKEN = bool(
    automatic_special_count == 0 and tokenizer.bos_token is not None
)
if automatic_special_count == 0 and not MINICONS_BOS_TOKEN:
    raise RuntimeError(
        "The tokenizer adds no initial special token and exposes no BOS token. "
        "Sentence-initial causal scoring would be undefined."
    )

MINICONS_SCORING_KWARGS = {
    "bos_token": MINICONS_BOS_TOKEN,
    "eos_token": MINICONS_EOS_TOKEN,
    "chat": MINICONS_CHAT_MODE,
}

print("Input device:", INPUT_DEVICE)
print("minicons device:", MINICONS_DEVICE)
print("Automatic special tokens:", automatic_special_count)
print("Manual minicons BOS:", MINICONS_BOS_TOKEN)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocab size:", len(tokenizer))
if int(model.config.vocab_size) != len(tokenizer):
    raise RuntimeError(
        f"Model/tokenizer vocabulary mismatch: {model.config.vocab_size} != {len(tokenizer)}"
    )
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


LOG2_E = math.log(2.0)


def raw_token_ids(text):
    text = clean_text_value(text)
    if not text:
        return []
    return tokenizer(text, add_special_tokens=False).input_ids


def truncate_context_text(context_text, target_token_count=0):
    """Left-truncate raw context tokens; minicons still controls special tokens."""
    context_text = clean_text_value(context_text)
    context_ids = raw_token_ids(context_text)

    special_budget = tokenizer.num_special_tokens_to_add(pair=False)
    if MINICONS_BOS_TOKEN:
        special_budget += 1

    max_context_length = (
        MAX_CONTEXT_TOKENS - special_budget - int(target_token_count)
    )
    if max_context_length <= 0:
        return ""
    if len(context_ids) <= max_context_length:
        return context_text

    retained = context_ids[-max_context_length:]
    return tokenizer.decode(
        retained,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


def minicons_formatted_text(text):
    """Reproduce IncrementalLMScorer.encode formatting without calling
    minicons 0.3.38's broken add_special_tokens helper.

    In minicons 0.3.38, IncrementalLMScorer.add_special_tokens contains a
    nested-function argument bug. IncrementalLMScorer.encode itself uses the
    intended behavior: prepend tokenizer.bos_token when bos_token=True and
    append tokenizer.eos_token when eos_token=True.
    """
    formatted = clean_text_value(text)

    if MINICONS_BOS_TOKEN:
        if tokenizer.bos_token is None:
            raise RuntimeError(
                "MINICONS_BOS_TOKEN=True but tokenizer.bos_token is missing."
            )
        formatted = tokenizer.bos_token + formatted

    if MINICONS_EOS_TOKEN:
        if tokenizer.eos_token is None:
            raise RuntimeError(
                "MINICONS_EOS_TOKEN=True but tokenizer.eos_token is missing."
            )
        formatted = formatted + tokenizer.eos_token

    return formatted


def encode_with_minicons_specials(text):
    """Match IncrementalLMScorer.encode and retain offset mappings for audit."""
    text = clean_text_value(text)
    formatted = minicons_formatted_text(text)
    manual_prefix_length = len(formatted) - len(text)

    encoded = tokenizer(
        [formatted],
        return_tensors="pt",
        padding=True,
        add_special_tokens=True,
        return_offsets_mapping=True,
    )
    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded.pop("token_type_ids", None)

    # Verify that this is byte-for-byte the same ID sequence produced by minicons.
    reference = lm_scorer.encode(
        [text],
        **MINICONS_SCORING_KWARGS,
    )
    if not torch.equal(encoded["input_ids"].cpu(), reference["input_ids"].cpu()):
        raise RuntimeError(
            "Custom offset-aware encoding does not match minicons encoding."
        )

    if MINICONS_DEVICE != "auto":
        encoded = encoded.to(INPUT_DEVICE)

    return encoded, offsets, manual_prefix_length


def active_sequence(encoded, offsets):
    attention_mask = encoded["attention_mask"][0].bool()
    active_positions = (
        torch.nonzero(attention_mask, as_tuple=False)
        .flatten()
        .tolist()
    )
    active_ids = encoded["input_ids"][0, attention_mask].tolist()
    active_offsets = [offsets[position] for position in active_positions]
    special_mask = tokenizer.get_special_tokens_mask(
        active_ids,
        already_has_special_tokens=True,
    )
    return active_positions, active_ids, active_offsets, special_mask


def text_token_positions(text, active_offsets, special_mask, manual_prefix_length):
    text_start = manual_prefix_length
    text_end = manual_prefix_length + len(text)
    return [
        position
        for position, ((start, end), is_special) in enumerate(
            zip(active_offsets, special_mask)
        )
        if end > start
        and end > text_start
        and start < text_end
    ]


def last_context_active_position(text, active_offsets, special_mask, manual_prefix_length):
    content_positions = text_token_positions(
        text,
        active_offsets,
        special_mask,
        manual_prefix_length,
    )
    if content_positions:
        return content_positions[-1]

    # Empty lexical context: use the first leading special token, never a terminal SEP/EOS.
    leading_specials = [
        position
        for position, is_special in enumerate(special_mask)
        if is_special
    ]
    if not leading_specials:
        raise RuntimeError(
            "No usable leading special token is available for empty-context scoring."
        )
    return leading_specials[0]


def entropy_bits_from_logits(logits):
    logits = logits.float()
    log_probs_nats = torch.log_softmax(logits, dim=-1)
    probabilities = torch.exp(log_probs_nats)
    entropy_nats = -torch.sum(probabilities * log_probs_nats)
    return float((entropy_nats / LOG2_E).detach().cpu())


@torch.inference_mode()
def lexical_and_word_entropy_bits(context_text, word):
    word = clean_text_value(word)
    target_count = len(raw_token_ids(word))
    if target_count == 0:
        return np.nan, np.nan

    context_text = truncate_context_text(
        context_text,
        target_token_count=target_count,
    )
    full_text = context_text + word
    encoded, offsets, manual_prefix_length = encode_with_minicons_specials(
        full_text
    )
    active_positions, active_ids, active_offsets, special_mask = active_sequence(
        encoded,
        offsets,
    )

    target_start = manual_prefix_length + len(context_text)
    target_end = target_start + len(word)
    target_active_positions = [
        position
        for position, ((start, end), is_special) in enumerate(
            zip(active_offsets, special_mask)
        )
        if end > start
        and end > target_start
        and start < target_end
    ]
    if not target_active_positions:
        return np.nan, np.nan

    # A token spanning both context and target would make lexical attribution ambiguous.
    straddling = [
        (active_offsets[position], active_ids[position])
        for position in target_active_positions
        if active_offsets[position][0] < target_start
        or active_offsets[position][1] > target_end
    ]
    if straddling:
        raise RuntimeError(
            "Tokenizer produced context-target boundary-straddling tokens: "
            f"{straddling[:5]}"
        )

    model_positions = [
        active_positions[position]
        for position in target_active_positions
    ]
    if any(position <= 0 for position in model_positions):
        raise RuntimeError("A lexical target token has no preceding causal state.")

    target_ids = torch.as_tensor(
        [active_ids[position] for position in target_active_positions],
        dtype=torch.long,
    )

    logits = model(**encoded).logits[0].float()
    prediction_positions = torch.as_tensor(
        [position - 1 for position in model_positions],
        dtype=torch.long,
        device=logits.device,
    )
    target_ids = target_ids.to(logits.device)

    target_distributions = torch.log_softmax(
        logits.index_select(0, prediction_positions),
        dim=-1,
    )
    token_logprobs_nats = target_distributions[
        torch.arange(len(target_ids), device=logits.device),
        target_ids,
    ]
    lexical_surprisal_bits = float(
        (-token_logprobs_nats.sum() / LOG2_E).detach().cpu()
    )

    probabilities = torch.exp(target_distributions)
    token_entropies_bits = -torch.sum(
        probabilities * target_distributions,
        dim=-1,
    ) / LOG2_E
    word_internal_entropy_bits = float(
        token_entropies_bits.sum().detach().cpu()
    )
    return lexical_surprisal_bits, word_internal_entropy_bits


@torch.inference_mode()
def entropy_after_text_bits(text):
    text = truncate_context_text(text)
    encoded, offsets, manual_prefix_length = encode_with_minicons_specials(text)
    active_positions, _ids, active_offsets, special_mask = active_sequence(
        encoded,
        offsets,
    )
    last_active = last_context_active_position(
        text,
        active_offsets,
        special_mask,
        manual_prefix_length,
    )
    model_position = active_positions[last_active]
    logits = model(**encoded).logits[0, model_position]
    return entropy_bits_from_logits(logits)


class CifuPOSLexicon:
    def __init__(self, cifu_path, valid_pos_set):
        self.valid_pos_set = {
            canonical_pos_tag(pos)
            for pos in valid_pos_set
            if valid_pos_for_pos_surprisal(pos)
        }
        self.words = []
        self.poses = []
        self.sequences = []
        self.groups = {}
        self.ancestor_pairs_by_length = {}
        self._load(cifu_path)

    def _load(self, cifu_path):
        cifu = pd.read_csv(cifu_path)
        required = {
            "cifu_word_clean",
            "pycantonese_pos",
            "cifu_pos_candidate_valid",
        }
        missing = sorted(required - set(cifu.columns))
        if missing:
            raise ValueError(f"Cifu POS file is missing columns: {missing}")

        valid_flag = cifu["cifu_pos_candidate_valid"]
        if pd.api.types.is_bool_dtype(valid_flag):
            valid_mask = valid_flag.fillna(False)
        else:
            valid_mask = (
                valid_flag.fillna("").astype(str).str.strip().str.lower()
                .isin({"true", "1", "yes", "y"})
            )
        cifu = cifu.loc[valid_mask].copy()
        cifu["candidate_word"] = cifu["cifu_word_clean"].map(li_text_normalizer)
        cifu["candidate_pos"] = cifu["pycantonese_pos"].map(canonical_pos_tag)
        cifu = cifu[
            cifu["candidate_word"].ne("")
            & cifu["candidate_pos"].isin(self.valid_pos_set)
        ]
        cifu = cifu.drop_duplicates("candidate_word")

        unk_id = tokenizer.unk_token_id
        special_ids = set(tokenizer.all_special_ids)
        seen_sequences = set()

        for word, pos in tqdm(
            cifu[["candidate_word", "candidate_pos"]].itertuples(index=False),
            total=len(cifu),
            desc="Tokenizing normalized Cifu POS candidates",
        ):
            token_ids = tuple(raw_token_ids(word))
            if not token_ids:
                continue
            if unk_id is not None and unk_id in token_ids:
                continue
            if any(token_id in special_ids for token_id in token_ids):
                continue
            if token_ids in seen_sequences:
                continue

            seen_sequences.add(token_ids)
            self.words.append(word)
            self.poses.append(pos)
            self.sequences.append(token_ids)

        if not self.sequences:
            raise ValueError("No valid tokenized Cifu POS candidates were found.")

        self.poses = np.asarray(self.poses, dtype=object)
        self.lengths = np.asarray(
            [len(sequence) for sequence in self.sequences],
            dtype=np.int32,
        )
        self.max_token_length = int(self.lengths.max())

        for length in sorted(set(self.lengths.tolist())):
            indices = np.flatnonzero(self.lengths == length)
            matrix = np.asarray(
                [self.sequences[index] for index in indices],
                dtype=np.int64,
            )
            self.groups[length] = (indices, matrix)

        terminal_by_sequence = {
            sequence: index
            for index, sequence in enumerate(self.sequences)
        }
        for length, (indices, _matrix) in self.groups.items():
            descendants = []
            ancestors = []
            for index in indices:
                sequence = self.sequences[index]
                for prefix_length in range(1, len(sequence)):
                    ancestor = terminal_by_sequence.get(
                        sequence[:prefix_length]
                    )
                    if ancestor is not None:
                        descendants.append(index)
                        ancestors.append(ancestor)
            self.ancestor_pairs_by_length[length] = (
                np.asarray(descendants, dtype=np.int64),
                np.asarray(ancestors, dtype=np.int64),
            )

        print("Cifu candidate words:", len(self.words))
        print("Cifu POS counts:")
        print(pd.Series(self.poses).value_counts())
        print("Cifu token-length counts:")
        print(pd.Series(self.lengths).value_counts().sort_index())

    def longest_match_exact_probabilities(self, joint_probabilities):
        exact = np.asarray(joint_probabilities, dtype=np.float64).copy()

        for length in sorted(self.groups, reverse=True):
            descendants, ancestors = self.ancestor_pairs_by_length[length]
            if len(descendants):
                np.add.at(exact, ancestors, -exact[descendants])

        negative_tolerance = 1e-4
        materially_negative = exact < -negative_tolerance
        if materially_negative.any():
            raise ValueError(
                "Longest-match subtraction produced materially negative "
                f"probabilities: count={int(materially_negative.sum())}, "
                f"minimum={float(exact.min())}"
            )
        return np.clip(exact, 0.0, None)


observed_pos_set = {
    canonical_pos_tag(pos)
    for pos in full_df["_llm_pos_for_calc"].dropna().tolist()
    if valid_pos_for_pos_surprisal(pos)
}
print("Observed POS tags for POS surprisal:", sorted(observed_pos_set))
CIFU_LEXICON = CifuPOSLexicon(cifu_pos_path, observed_pos_set)


def repeat_past_key_values(past_key_values, batch_size):
    repeated = copy.deepcopy(past_key_values)
    if hasattr(repeated, "batch_repeat_interleave"):
        repeated.batch_repeat_interleave(batch_size)
        return repeated

    return tuple(
        tuple(
            state.repeat_interleave(batch_size, dim=0)
            for state in layer
        )
        for layer in repeated
    )


@torch.inference_mode()
def cifu_joint_probabilities(context_text):
    context_text = truncate_context_text(
        context_text,
        target_token_count=CIFU_LEXICON.max_token_length,
    )
    encoded, offsets, manual_prefix_length = encode_with_minicons_specials(
        context_text
    )
    active_positions, _ids, active_offsets, special_mask = active_sequence(
        encoded,
        offsets,
    )
    last_active = last_context_active_position(
        context_text,
        active_offsets,
        special_mask,
        manual_prefix_length,
    )
    last_model_position = active_positions[last_active]

    # Remove any terminal SEP/EOS from the cached prefix, as Li's token-score
    # post-processing excluded terminal special tokens from lexical scores.
    prefix_input_ids = encoded["input_ids"][:, : last_model_position + 1]
    prefix_attention_mask = encoded["attention_mask"][:, : last_model_position + 1]

    context_output = model(
        input_ids=prefix_input_ids,
        attention_mask=prefix_attention_mask,
        use_cache=True,
    )
    first_log_probs = torch.log_softmax(
        context_output.logits[0, -1, :].float(),
        dim=-1,
    )
    context_past = context_output.past_key_values
    joint_log_probs = np.full(
        len(CIFU_LEXICON.words),
        -np.inf,
        dtype=np.float64,
    )

    for length, (indices, matrix) in CIFU_LEXICON.groups.items():
        first_ids = torch.as_tensor(
            matrix[:, 0],
            dtype=torch.long,
            device=first_log_probs.device,
        )
        first_scores = first_log_probs.index_select(0, first_ids)

        if length == 1:
            joint_log_probs[indices] = first_scores.detach().cpu().numpy()
            continue

        prefix_matrix = matrix[:, :-1]
        final_targets = matrix[:, -1]
        unique_prefixes, inverse = np.unique(
            prefix_matrix,
            axis=0,
            return_inverse=True,
        )
        order = np.argsort(inverse)
        sorted_inverse = inverse[order]
        bounds = np.searchsorted(
            sorted_inverse,
            np.arange(len(unique_prefixes) + 1),
        )

        for start in range(
            0,
            len(unique_prefixes),
            CIFU_CANDIDATE_BATCH_SIZE,
        ):
            stop = min(
                start + CIFU_CANDIDATE_BATCH_SIZE,
                len(unique_prefixes),
            )
            batch_prefixes = torch.as_tensor(
                unique_prefixes[start:stop],
                dtype=torch.long,
                device=INPUT_DEVICE,
            )
            repeated_past = repeat_past_key_values(
                context_past,
                int(batch_prefixes.shape[0]),
            )
            output = model(
                input_ids=batch_prefixes,
                past_key_values=repeated_past,
                use_cache=False,
            )
            continuation_log_probs = torch.log_softmax(
                output.logits.float(),
                dim=-1,
            )

            prefix_len = int(batch_prefixes.shape[1])
            if prefix_len > 1:
                internal_targets = batch_prefixes[:, 1:].unsqueeze(-1)
                internal_scores = continuation_log_probs[:, :-1, :].gather(
                    2,
                    internal_targets,
                ).squeeze(-1).sum(dim=1)
            else:
                internal_scores = torch.zeros(
                    int(batch_prefixes.shape[0]),
                    dtype=continuation_log_probs.dtype,
                    device=continuation_log_probs.device,
                )

            candidate_positions = order[bounds[start]:bounds[stop]]
            local_prefix_ids = inverse[candidate_positions] - start
            local_prefix_ids_t = torch.as_tensor(
                local_prefix_ids,
                dtype=torch.long,
                device=continuation_log_probs.device,
            )
            final_targets_t = torch.as_tensor(
                final_targets[candidate_positions],
                dtype=torch.long,
                device=continuation_log_probs.device,
            )
            candidate_positions_t = torch.as_tensor(
                candidate_positions,
                dtype=torch.long,
                device=first_scores.device,
            )
            final_scores = continuation_log_probs[
                local_prefix_ids_t,
                -1,
                final_targets_t,
            ]
            continuation_scores = (
                internal_scores[local_prefix_ids_t] + final_scores
            )
            total_scores = (
                first_scores.index_select(0, candidate_positions_t)
                + continuation_scores.to(first_scores.device)
            )
            joint_log_probs[indices[candidate_positions]] = (
                total_scores.detach().cpu().numpy()
            )

    # Candidate probabilities are exponentiated from natural-log model scores.
    # POS surprisal itself is converted to base 2 below.
    return np.exp(joint_log_probs)


@torch.inference_mode()
def pos_surprisal_bits(context_text, observed_pos):
    observed_pos = canonical_pos_tag(observed_pos)
    if (
        not valid_pos_for_pos_surprisal(observed_pos)
        or observed_pos not in CIFU_LEXICON.valid_pos_set
    ):
        return np.nan

    joint = cifu_joint_probabilities(context_text)
    exact = CIFU_LEXICON.longest_match_exact_probabilities(joint)
    total_mass = float(exact.sum())
    if total_mass <= 0:
        return np.nan
    if total_mass > 1.0 + 1e-4:
        raise ValueError(
            f"Cifu longest-match probability mass exceeds 1: {total_mass}"
        )

    true_pos_mass = float(
        exact[CIFU_LEXICON.poses == observed_pos].sum()
    )
    true_pos_probability = true_pos_mass / total_mass
    if true_pos_probability <= 0:
        return np.nan

    return -math.log2(max(true_pos_probability, EPS))


# Fail fast on API or scoring incompatibilities before the 281-sentence run.
required_torch_api = [
    "arange",
    "as_tensor",
    "equal",
    "exp",
    "inference_mode",
    "log_softmax",
    "nonzero",
    "sum",
    "zeros",
]
missing_torch_api = [
    name for name in required_torch_api if not hasattr(torch, name)
]
if missing_torch_api:
    raise RuntimeError(
        f"Installed torch is missing required APIs: {missing_torch_api}"
    )

_preflight_rows = df.loc[df["_llm_metric_row_included"]]
if _preflight_rows.empty:
    raise RuntimeError("No lexical rows are available for metric extraction.")

_preflight_row = _preflight_rows.iloc[0]
_preflight_context = clean_text_value(_preflight_row["_llm_pre_context"])
_preflight_word = clean_text_value(_preflight_row["_llm_word_for_calc"])
_preflight_pos = clean_text_value(_preflight_row["_llm_pos_for_calc"])

_preflight_lexical, _preflight_internal_entropy = (
    lexical_and_word_entropy_bits(
        _preflight_context,
        _preflight_word,
    )
)
_preflight_entropy_before = entropy_after_text_bits(_preflight_context)
_preflight_entropy_after = entropy_after_text_bits(
    _preflight_context + _preflight_word
)
_preflight_pos_surprisal = pos_surprisal_bits(
    _preflight_context,
    _preflight_pos,
)

for _name, _value in {
    "lexical_surprisal": _preflight_lexical,
    "word_internal_entropy": _preflight_internal_entropy,
    "entropy_before": _preflight_entropy_before,
    "entropy_after": _preflight_entropy_after,
}.items():
    if not np.isfinite(_value):
        raise RuntimeError(
            f"Preflight produced a non-finite {_name}: {_value}"
        )

print(
    "Preflight passed:",
    {
        "word": _preflight_word,
        "lexical_surprisal_bits": _preflight_lexical,
        "word_internal_entropy_bits": _preflight_internal_entropy,
        "entropy_before_bits": _preflight_entropy_before,
        "entropy_after_bits": _preflight_entropy_after,
        "pos_surprisal_bits": _preflight_pos_surprisal,
    },
)

METRIC_COLUMNS_EXTRACTED = [
    f"{MODEL_PREFIX}lexical_surprisal",
    f"{MODEL_PREFIX}pos_surprisal",
    f"{MODEL_PREFIX}entropy",
    f"{MODEL_PREFIX}entropy_before",
    f"{MODEL_PREFIX}entropy_after",
    f"{MODEL_PREFIX}entropy_reduction",
]


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=False, encoding="utf-8-sig")
    os.replace(temporary, path)


def sentence_checkpoint_path(sentence_id, group):
    signature_payload = {
            "checkpoint_version": EXTRACTION_CHECKPOINT_VERSION,
            "model_id": MODEL_ID,
            "model_commit": getattr(model.config, "_commit_hash", None),
            "tokenizer_id": TOKENIZER_ID,
            "source_sha256": SOURCE_INPUT_SHA256,
            "cifu_sha256": CIFU_FILE_SHA256,
            "sentence": str(sentence_id),
            "indices": [int(value) for value in group.index],
            "surfaces": group["_li_normalized_surface"].astype(str).tolist(),
            "targets": group["_llm_word_for_calc"].astype(str).tolist(),
            "poses": group["_llm_pos_for_calc"].astype(str).tolist(),
    }
    sentence_text = "".join(
        group["_li_normalized_surface"].fillna("").astype(str).tolist()
    )
    unk_id = tokenizer.unk_token_id
    if unk_id is not None and int(unk_id) in raw_token_ids(sentence_text):
        # Only sentences affected by the old [UNK]-as-special bug get a new
        # signature; all unaffected sentence checkpoints remain reusable.
        signature_payload["unk_offset_content_policy"] = UNK_OFFSET_CONTENT_POLICY
    signature = json.dumps(
        signature_payload,
        ensure_ascii=False,
        sort_keys=True,
    )
    digest = hashlib.sha256(signature.encode("utf-8")).hexdigest()[:16]
    return CHECKPOINT_DIR / f"sentence_{sentence_id}_{digest}.csv"


def valid_sentence_checkpoint(frame, group):
    expected_columns = ["index", *METRIC_COLUMNS_EXTRACTED]
    if frame.columns.tolist() != expected_columns:
        return False
    expected_indices = [int(value) for value in group.index]
    actual_indices = pd.to_numeric(frame["index"], errors="coerce").tolist()
    return len(frame) == len(group) and actual_indices == expected_indices


metric_rows = []
allowed_for_scoring = SCORABLE_ALIGNMENT
sentence_status_rows = []

for _sentence_id, group in tqdm(
    df.groupby(SENTENCE_COL, sort=False),
    desc=f"Computing {MODEL_PREFIX} four LLM metrics",
):
    group = group.sort_values(ORDER_COL, kind="stable")
    checkpoint_path = sentence_checkpoint_path(_sentence_id, group)
    checkpoint_frame = None
    if checkpoint_path.exists():
        try:
            candidate = pd.read_csv(checkpoint_path)
        except Exception as exc:
            print(f"Ignoring unreadable checkpoint {checkpoint_path}: {exc}")
        else:
            if valid_sentence_checkpoint(candidate, group):
                checkpoint_frame = candidate

    if checkpoint_frame is not None:
        metric_rows.extend(checkpoint_frame.to_dict("records"))
        sentence_status_rows.append({
            "sentence": _sentence_id,
            "status": "checkpoint",
            "rows": int(len(group)),
            "checkpoint_file": str(checkpoint_path),
        })
        continue

    sentence_metric_rows = []
    prefix_text = ""
    for index, row in group.iterrows():
        surface = clean_text_value(row["_li_normalized_surface"])
        pre_context = clean_text_value(row["_llm_pre_context"])
        word = clean_text_value(row["_llm_word_for_calc"])
        post_context = clean_text_value(row["_llm_post_context"])
        pos = clean_text_value(row["_llm_pos_for_calc"])
        alignment_status = clean_text_value(row["_llm_alignment_status"])

        scoring_context = prefix_text + pre_context
        can_score = bool(row["_llm_metric_row_included"])

        if not can_score:
            lexical = np.nan
            pos_s = np.nan
            entropy = np.nan
            entropy_before = np.nan
            entropy_after = np.nan
            entropy_reduction = np.nan
        else:
            lexical, entropy = lexical_and_word_entropy_bits(scoring_context, word)
            pos_s = pos_surprisal_bits(scoring_context, pos)
            entropy_before = entropy_after_text_bits(scoring_context)
            entropy_after = entropy_after_text_bits(scoring_context + word)
            entropy_reduction = entropy_before - entropy_after

        record = {
            "index": int(index),
            f"{MODEL_PREFIX}lexical_surprisal": lexical,
            f"{MODEL_PREFIX}pos_surprisal": pos_s,
            f"{MODEL_PREFIX}entropy": entropy,
            f"{MODEL_PREFIX}entropy_before": entropy_before,
            f"{MODEL_PREFIX}entropy_after": entropy_after,
            f"{MODEL_PREFIX}entropy_reduction": entropy_reduction,
        }
        sentence_metric_rows.append(record)
        metric_rows.append(record)

        # Preserve the exact Li-normalized punctuated context used by JED351.
        prefix_text += surface
        if alignment_status in allowed_for_scoring:
            reconstructed = pre_context + word + post_context
            if reconstructed != surface:
                raise RuntimeError(
                    "Surface reconstruction failed after alignment: "
                    f"{reconstructed!r} != {surface!r}"
                )

    atomic_write_csv(pd.DataFrame(sentence_metric_rows), checkpoint_path)
    sentence_status_rows.append({
        "sentence": _sentence_id,
        "status": "computed",
        "rows": int(len(group)),
        "checkpoint_file": str(checkpoint_path),
    })

sentence_status_df = pd.DataFrame(sentence_status_rows)
print("Sentence statuses:", sentence_status_df["status"].value_counts().to_dict())

metrics_df = pd.DataFrame(metric_rows).set_index("index").sort_index()

audit_rename = {
    "_li_normalized_surface": "li_normalized_ia",
    "_llm_word_for_calc": "llm_lexical_target",
    "_llm_pre_context": "llm_pre_target_context",
    "_llm_post_context": "llm_post_target_context",
    "_llm_alignment_status": "llm_text_alignment_status",
    "_llm_metric_row_included": "llm_metric_row_included",
    "_llm_pos_for_calc": "llm_pos_target",
}
audit_df = df[list(audit_rename)].rename(columns=audit_rename)

out = pd.concat(
    [
        df.drop(columns=list(audit_rename)),
        audit_df,
        metrics_df,
    ],
    axis=1,
)

metric_cols = [
    column
    for column in out.columns
    if column.startswith(MODEL_PREFIX)
]
print("Metric unit: bits (base-2)")
print("Metric columns:", metric_cols)
print("Rows included:", int(out["llm_metric_row_included"].sum()))
print("Missing counts:")
print(out[metric_cols].isna().sum())
display(out.head())


suffix = "" if RUN_LIMIT_ROWS is None else f"_limit{RUN_LIMIT_ROWS}"
stem = (
    f"NEW_trad_NR_with_{MODEL_PREFIX.rstrip('_')}_four_llm_metrics"
    "_li_normalized_punctuated_context_minicons_bits"
    "_cifu_longest_match"
)
csv_path = OUTPUT_DIR / f"{stem}{suffix}.csv"
xlsx_path = OUTPUT_DIR / f"{stem}{suffix}.xlsx"
manifest_path = OUTPUT_DIR / f"{stem}{suffix}_manifest.json"

out.to_csv(csv_path, index=False, encoding="utf-8-sig")
out.to_excel(xlsx_path, index=False)

manifest = {
    "model_id": MODEL_ID,
    "model_prefix": MODEL_PREFIX,
    "resolved_model_commit": getattr(model.config, "_commit_hash", None),
    "tokenizer_id": TOKENIZER_ID,
    "tokenizer_class": tokenizer.__class__.__name__,
    "checkpoint_version": EXTRACTION_CHECKPOINT_VERSION,
    "unk_offset_content_policy": UNK_OFFSET_CONTENT_POLICY,
    "source_input_sha256": SOURCE_INPUT_SHA256,
    "cifu_file_sha256": CIFU_FILE_SHA256,
    "input_filename": INPUT_FILENAME,
    "cifu_pos_filename": CIFU_POS_FILENAME,
    "sheet_name": SHEET_NAME,
    "rows": int(len(out)),
    "metric_rows_included": int(out["llm_metric_row_included"].sum()),
    "alignment_status_counts": {
        str(key): int(value)
        for key, value in out["llm_text_alignment_status"]
        .value_counts(dropna=False)
        .items()
    },
    "log_base": 2,
    "unit": "bits",
    "text_normalization": "Li et al. (2023) released text_normalizer",
    "punctuation_policy": (
        "Li-normalized punctuation is retained in context; lexical metrics "
        "exclude attached punctuation; trailing punctuation enters only the "
        "context of subsequent words."
    ),
    "special_token_policy": (
        "minicons IncrementalLMScorer formatting; terminal SEP/EOS excluded "
        "when defining next-token states."
    ),
    "minicons_manual_bos": bool(MINICONS_BOS_TOKEN),
    "minicons_version": importlib.metadata.version("minicons"),
    "transformers_version": importlib.metadata.version("transformers"),
    "torch_version": torch.__version__,
    "max_context_tokens": int(MAX_CONTEXT_TOKENS),
    "run_limit_rows": RUN_LIMIT_ROWS,
    "metric_columns": metric_cols,
}
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(csv_path)
print(xlsx_path)
print(manifest_path)

sentence_status_path = OUTPUT_DIR / "sentence_checkpoint_status.csv"
sentence_status_df.to_csv(sentence_status_path, index=False, encoding="utf-8-sig")
print("Extraction files and checkpoints retained under:", OUTPUT_DIR)


## 3. Enforce exact JED351 sample and map metrics to NR/TSR


In [ ]:

# Map NR text-derived metrics to both tasks by canonical text position.
base_modeling = pd.read_csv(base_modeling_path, low_memory=False)
EXPECTED_ROWS = 10089
EXPECTED_TASK_COUNTS = {"NR": 5046, "TSR": 5043}
KEY_COLUMNS = ["WORD", "SENTENCE", "n_in_sent"]
PRIMARY_METRICS = [
    f"{MODEL_PREFIX}lexical_surprisal",
    f"{MODEL_PREFIX}pos_surprisal",
    f"{MODEL_PREFIX}entropy_before",
    f"{MODEL_PREFIX}entropy_reduction",
]
ALL_MODEL_METRICS = [
    f"{MODEL_PREFIX}lexical_surprisal",
    f"{MODEL_PREFIX}pos_surprisal",
    f"{MODEL_PREFIX}entropy",
    f"{MODEL_PREFIX}entropy_before",
    f"{MODEL_PREFIX}entropy_after",
    f"{MODEL_PREFIX}entropy_reduction",
]
SOURCE_AUDIT_COLUMNS = [
    "li_normalized_ia",
    "llm_lexical_target",
    "llm_pre_target_context",
    "llm_post_target_context",
    "llm_text_alignment_status",
    "llm_metric_row_included",
    "llm_pos_target",
]

required_base = {
    "canonical_row_id", "TASK", "IA_LABEL", *KEY_COLUMNS,
    "pure_token", *[f"jed351_{name}" for name in (
        "lexical_surprisal", "pos_surprisal", "entropy_before", "entropy_reduction"
    )],
}
missing_base = sorted(required_base - set(base_modeling.columns))
if missing_base:
    raise KeyError(f"JED351 modeling base is missing: {missing_base}")
if len(base_modeling) != EXPECTED_ROWS:
    raise ValueError(f"Expected {EXPECTED_ROWS:,} JED351 rows, found {len(base_modeling):,}.")
if base_modeling["TASK"].value_counts().to_dict() != EXPECTED_TASK_COUNTS:
    raise ValueError("The JED351 task counts do not match the reference sample.")
if base_modeling[["canonical_row_id", "TASK"]].duplicated().any():
    raise ValueError("canonical_row_id must be unique within each task.")

source_metrics = out[
    [*KEY_COLUMNS, "IA_LABEL", *ALL_MODEL_METRICS, *SOURCE_AUDIT_COLUMNS]
].copy()
for column in KEY_COLUMNS:
    source_metrics[column] = pd.to_numeric(source_metrics[column], errors="raise").astype(int)
if source_metrics.duplicated(KEY_COLUMNS).any():
    raise ValueError("The extracted NR key columns are not unique.")

base_nr = base_modeling.loc[base_modeling["TASK"].eq("NR"), [
    "canonical_row_id", *KEY_COLUMNS, "IA_LABEL"
]].copy()
for column in KEY_COLUMNS:
    base_nr[column] = pd.to_numeric(base_nr[column], errors="raise").astype(int)

aligned_nr = base_nr.merge(
    source_metrics,
    on=KEY_COLUMNS,
    how="left",
    validate="one_to_one",
    suffixes=("_base", "_metric"),
    indicator=True,
)
if not aligned_nr["_merge"].eq("both").all():
    raise ValueError("Some JED351 reference NR rows did not match the CKIP extraction.")
label_match = (
    aligned_nr["IA_LABEL_base"].fillna("").astype(str)
    == aligned_nr["IA_LABEL_metric"].fillna("").astype(str)
)
if not label_match.all():
    raise ValueError("IA_LABEL differs between the CKIP source and JED351 reference rows.")

audit_rename = {column: f"ckip_{column}" for column in SOURCE_AUDIT_COLUMNS}
metric_map = aligned_nr[
    ["canonical_row_id", *ALL_MODEL_METRICS, *SOURCE_AUDIT_COLUMNS]
].rename(columns=audit_rename)
if not metric_map["canonical_row_id"].is_unique:
    raise ValueError("The CKIP canonical metric map is not unique.")

drop_existing = [
    column for column in [*ALL_MODEL_METRICS, *audit_rename.values()]
    if column in base_modeling.columns
]
modeling = base_modeling.drop(columns=drop_existing).merge(
    metric_map,
    on="canonical_row_id",
    how="left",
    validate="many_to_one",
    sort=False,
)
numeric_metrics = modeling[PRIMARY_METRICS].apply(pd.to_numeric, errors="coerce")
finite_mask = np.isfinite(numeric_metrics.to_numpy(dtype=float)).all(axis=1)

PIPELINE_ROOT = OUTPUT_DIR.parent
FINAL_DATA_DIR = PIPELINE_ROOT / "data"
FINAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
missing_path = FINAL_DATA_DIR / "ckip_metrics_missing_on_jed351_reference_rows.csv"
if not finite_mask.all():
    missing_rows = modeling.loc[
        ~finite_mask,
        ["canonical_row_id", "TASK", *KEY_COLUMNS, "IA_LABEL", "pure_token", *PRIMARY_METRICS],
    ]
    missing_rows.to_csv(missing_path, index=False, encoding="utf-8-sig")
    raise ValueError(
        f"CKIP metrics are incomplete on {len(missing_rows)} JED351 reference rows. "
        f"See {missing_path}. The notebook stops to avoid changing the comparison sample."
    )
modeling[PRIMARY_METRICS] = numeric_metrics

CKIP_MODELING_PATH = (
    FINAL_DATA_DIR / "MCFIX_NR_TSR_CKIP_GPT2_tiny_JED351_matched_complete_cases.csv"
)
modeling.to_csv(CKIP_MODELING_PATH, index=False, encoding="utf-8-sig")
alignment_audit = {
    "base_file": str(base_modeling_path),
    "base_sha256": sha256_input_file(base_modeling_path),
    "output_file": str(CKIP_MODELING_PATH),
    "output_rows": int(len(modeling)),
    "task_counts": modeling["TASK"].value_counts().to_dict(),
    "reference_sample": "exact current JED351 complete-case sample",
    "reference_jed351_rows": EXPECTED_ROWS,
    "all_ckip_primary_metrics_finite": True,
    "metric_mapping": "NR metrics mapped to NR and TSR by canonical_row_id",
}
(FINAL_DATA_DIR / "ckip_jed351_matched_alignment_audit.json").write_text(
    json.dumps(alignment_audit, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Strict JED351-matched modeling CSV:", CKIP_MODELING_PATH)
print(json.dumps(alignment_audit, ensure_ascii=False, indent=2))


## 4. Release model GPU memory


In [ ]:

# Release the language model before CatBoost uses the GPU.
del model, tokenizer, lm_scorer, CIFU_LEXICON
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("GPU memory after release (GB):", round(torch.cuda.memory_allocated() / 1024**3, 3))


## 5. Run JED351-matched regression and ablations


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import platform
import shutil
import time
from pathlib import Path

import catboost
import numpy as np
import pandas as pd
import scipy
import sklearn
from catboost import CatBoostRegressor
from scipy.stats import pearsonr, spearmanr, t as student_t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from tqdm.auto import tqdm


# Prespecified robustness design.
SPLIT_SEEDS = (42, 43, 44, 45, 46)
N_SPLITS = 5
CATBOOST_RANDOM_SEED = 10
CATBOOST_ITERATIONS = 1000
TASK_TYPE = "GPU"                 # Use "CPU" for exact CPU-mode replication.
GPU_DEVICES = "0"
CPU_THREAD_COUNT = 4
SAVE_EVERY_N_JOBS = 5

# Persistent output is strongly recommended for this long run.
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_DIR = (
    str(PIPELINE_ROOT / "regression")
)
LOCAL_OUTPUT_DIR = "/content/MCFIX_CKIP_GPT2_tiny_JED351_matched/regression"

# Leave blank to upload the supplied complete-case CSV when prompted.
INPUT_PATH = str(CKIP_MODELING_PATH)

TASKS = ("NR", "TSR")
TARGETS = {
    "FFD": "IA_FIRST_FIXATION_DURATION",
    "SFD": "IA_SECOND_FIXATION_DURATION",
    "TFD": "TOTAL_DURATION",
}
LLM_METRICS = (
    "lexical_surprisal",
    "pos_surprisal",
    "entropy_before",
    "entropy_reduction",
)
METRIC_COLUMNS = {
    metric: f"ckip_gpt2_tiny_{metric}" for metric in LLM_METRICS
}

print("Configuration loaded.")
print("Partition seeds:", SPLIT_SEEDS)
print("CatBoost task type:", TASK_TYPE)


In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
else:
    OUTPUT_DIR = Path(LOCAL_OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = OUTPUT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

if INPUT_PATH:
    input_path = Path(INPUT_PATH)
else:
    candidates = list(Path("/content").glob("*CKIP_GPT2_tiny*complete_cases*.csv"))
    if candidates:
        input_path = candidates[0]
    else:
        from google.colab import files

        print("Upload MCFIX_NR_TSR_NEW_CKIP_GPT2_tiny_complete_cases.csv")
        uploaded = files.upload()
        csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_names) != 1:
            raise ValueError("Upload exactly one CSV input file.")
        input_path = Path("/content") / csv_names[0]

if not input_path.exists():
    raise FileNotFoundError(input_path)

print("Input:", input_path)
print("Output:", OUTPUT_DIR)


In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def regression_metrics(observed, predicted) -> dict[str, float]:
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    mse = float(mean_squared_error(observed, predicted))
    if np.std(observed) > 1e-12 and np.std(predicted) > 1e-12:
        pearson = float(pearsonr(observed, predicted).statistic)
        spearman = float(spearmanr(observed, predicted).statistic)
    else:
        pearson = math.nan
        spearman = math.nan
    return {
        "mae": float(mean_absolute_error(observed, predicted)),
        "mse": mse,
        "rmse": float(math.sqrt(mse)),
        "r2": float(r2_score(observed, predicted)),
        "pearson": pearson,
        "spearman": spearman,
    }


def prepare_data(path: Path):
    data = pd.read_csv(path, low_memory=False)
    required = {
        "canonical_row_id", "TASK", "SENTENCE", "n_in_sent",
        "sentence_length_tokens", "pure_token", "upos*", "LD2ROOT",
        "LD2HEAD", "DEPTH2ROOT", "Neighbors_Num", "N_SYL", "Freq",
        *METRIC_COLUMNS.values(), *TARGETS.values(),
    }
    missing = sorted(required.difference(data.columns))
    if missing:
        raise KeyError(f"Required columns are missing: {missing}")

    data["TASK"] = data["TASK"].astype(str)
    if set(data["TASK"].dropna()) != set(TASKS):
        raise ValueError("TASK must contain exactly NR and TSR.")
    if data[["canonical_row_id", "TASK"]].duplicated().any():
        raise ValueError("canonical_row_id must be unique within each task.")

    data["TASK_indicator"] = data["TASK"].eq("TSR").astype(np.int8)
    data["Li_log_Freq"] = np.log(pd.to_numeric(data["Freq"]) + 1.0)
    data["Li_N_SYL"] = pd.to_numeric(data["N_SYL"])
    data["Li_Word_Position"] = (
        (pd.to_numeric(data["n_in_sent"]) + 1.0)
        / pd.to_numeric(data["sentence_length_tokens"])
    )

    ordered = data.sort_values(
        ["TASK", "SENTENCE", "n_in_sent"], kind="stable"
    ).copy()
    groups = ordered.groupby(["TASK", "SENTENCE"], sort=False, observed=True)
    data["raw_prev_freq"] = groups["Freq"].shift(1)
    data["raw_prev_n_syl"] = groups["N_SYL"].shift(1)
    data["Li_Prev_Freq"] = np.where(
        data["raw_prev_freq"].isna(),
        0.0,
        np.log(pd.to_numeric(data["raw_prev_freq"]) + 2.0),
    )
    data["Li_Prev_N_SYL"] = pd.to_numeric(
        data["raw_prev_n_syl"]
    ).fillna(0.0)

    data["upos_model"] = (
        data["upos*"].fillna("MISSING").astype(str).replace({"V": "VERB"})
    )
    pos_dummies = pd.get_dummies(
        data["upos_model"], prefix="CPOS", dtype=np.int8
    )
    data = pd.concat([data, pos_dummies], axis=1)

    blocks = {
        "TASK": ["TASK_indicator"],
        "POS": pos_dummies.columns.tolist(),
        "LD2ROOT": ["LD2ROOT"],
        "LD2HEAD": ["LD2HEAD"],
        "DEPTH2ROOT": ["DEPTH2ROOT"],
        "Neighbors_Num": ["Neighbors_Num"],
        "Frequency": ["Li_log_Freq"],
        "Syllable_Number": ["Li_N_SYL"],
        "Previous_Frequency": ["Li_Prev_Freq"],
        "Previous_Syllable_Number": ["Li_Prev_N_SYL"],
        "Word_Position": ["Li_Word_Position"],
    }

    numeric_columns = (
        [column for columns in blocks.values() for column in columns]
        + list(METRIC_COLUMNS.values())
        + list(TARGETS.values())
    )
    numeric = data[numeric_columns].apply(pd.to_numeric, errors="coerce")
    if numeric.isna().any().any():
        counts = numeric.isna().sum()
        raise ValueError(
            "Missing modeling values:\n" + counts[counts.gt(0)].to_string()
        )
    if not np.isfinite(numeric.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite modeling values were found.")
    data[numeric_columns] = numeric
    return data.reset_index(drop=True), blocks


def add_fold_assignments(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    canonical_ids = data["canonical_row_id"].drop_duplicates().to_numpy()

    for seed in SPLIT_SEEDS:
        pooled_folds = np.full(len(data), -1, dtype=np.int8)
        splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold, (_, test) in enumerate(splitter.split(data), start=1):
            pooled_folds[test] = fold
        data[f"pooled_fold_seed_{seed}"] = pooled_folds

        canonical_to_fold = {}
        splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        for fold, (_, test) in enumerate(
            splitter.split(canonical_ids), start=1
        ):
            for canonical_id in canonical_ids[test]:
                canonical_to_fold[canonical_id] = fold
        data[f"task_fold_seed_{seed}"] = (
            data["canonical_row_id"].map(canonical_to_fold).astype(np.int8)
        )

    fold_columns = [
        column for column in data.columns
        if column.startswith(("pooled_fold_seed_", "task_fold_seed_"))
    ]
    if data[fold_columns].lt(1).any().any():
        raise RuntimeError("Incomplete fold assignment.")

    paired = data.groupby("canonical_row_id")["TASK"].nunique()
    paired_ids = paired.loc[paired.eq(2)].index
    for seed in SPLIT_SEEDS:
        synchronized = (
            data.loc[data["canonical_row_id"].isin(paired_ids)]
            .groupby("canonical_row_id")[f"task_fold_seed_{seed}"]
            .nunique()
        )
        if not synchronized.eq(1).all():
            raise RuntimeError(f"NR/TSR task folds differ for seed {seed}.")
    return data


data, FEATURE_BLOCKS = prepare_data(input_path)
data = add_fold_assignments(data)

if len(data) != 10089:
    raise ValueError(f"Expected 10,089 complete-case rows, found {len(data):,}.")
expected_task_counts = {"NR": 5046, "TSR": 5043}
actual_task_counts = data["TASK"].value_counts().to_dict()
if actual_task_counts != expected_task_counts:
    raise ValueError(
        f"Unexpected task counts: {actual_task_counts}; "
        f"expected {expected_task_counts}."
    )

fold_export_columns = [
    "canonical_row_id", "TASK", "SENTENCE", "n_in_sent", "pure_token",
    *[
        column for column in data.columns
        if column.startswith(("pooled_fold_seed_", "task_fold_seed_"))
    ],
]
data[fold_export_columns].to_csv(
    TABLES_DIR / "fold_assignments.csv", index=False, encoding="utf-8-sig"
)

print(f"Prepared {len(data):,} rows:", actual_task_counts)
print("POS dummy columns:", len(FEATURE_BLOCKS["POS"]))


In [ ]:
def baseline_columns(blocks, pooled: bool) -> list[str]:
    block_names = list(blocks)
    if not pooled:
        block_names.remove("TASK")
    return [column for name in block_names for column in blocks[name]]


def build_specifications(blocks, pooled: bool):
    baseline = baseline_columns(blocks, pooled)
    all_metric_columns = list(METRIC_COLUMNS.values())
    specifications = {"baseline_no_llm": baseline}
    rows = [{
        "specification": "baseline_no_llm",
        "analysis_type": "baseline",
        "changed_block": "NONE",
        "n_features": len(baseline),
    }]

    specifications["all_four"] = baseline + all_metric_columns
    rows.append({
        "specification": "all_four",
        "analysis_type": "full_all_four",
        "changed_block": "NONE",
        "n_features": len(specifications["all_four"]),
    })

    for metric, column in METRIC_COLUMNS.items():
        name = f"add_{metric}"
        specifications[name] = baseline + [column]
        rows.append({
            "specification": name,
            "analysis_type": "single_metric_addition",
            "changed_block": metric,
            "n_features": len(specifications[name]),
        })

    for metric, column in METRIC_COLUMNS.items():
        name = f"all_four_minus_{metric}"
        specifications[name] = baseline + [
            value for value in all_metric_columns if value != column
        ]
        rows.append({
            "specification": name,
            "analysis_type": "leave_one_metric_out",
            "changed_block": metric,
            "n_features": len(specifications[name]),
        })

    for block_name, removed_columns in blocks.items():
        if not pooled and block_name == "TASK":
            continue
        removed = set(removed_columns)
        name = f"all_four_minus_{block_name}"
        specifications[name] = [
            column for column in baseline if column not in removed
        ] + all_metric_columns
        rows.append({
            "specification": name,
            "analysis_type": "leave_one_control_block_out",
            "changed_block": block_name,
            "n_features": len(specifications[name]),
        })
    return specifications, pd.DataFrame(rows)


DESIGNS = (
    ("pooled_with_TASK", "POOLED", True),
    ("task_specific", "NR", False),
    ("task_specific", "TSR", False),
)

SPECIFICATIONS = {}
specification_frames = []
for design, training_task, pooled in DESIGNS:
    specs, frame = build_specifications(FEATURE_BLOCKS, pooled)
    SPECIFICATIONS[(design, training_task)] = specs
    frame.insert(0, "training_task", training_task)
    frame.insert(0, "design", design)
    specification_frames.append(frame)

feature_specifications = pd.concat(specification_frames, ignore_index=True)
feature_specifications.to_csv(
    TABLES_DIR / "feature_specifications.csv",
    index=False,
    encoding="utf-8-sig",
)

jobs = []
for split_seed in SPLIT_SEEDS:
    for design, training_task, _ in DESIGNS:
        for specification, columns in SPECIFICATIONS[(design, training_task)].items():
            metadata = feature_specifications.loc[
                feature_specifications["design"].eq(design)
                & feature_specifications["training_task"].eq(training_task)
                & feature_specifications["specification"].eq(specification)
            ].iloc[0]
            for target, target_column in TARGETS.items():
                jobs.append({
                    "split_seed": split_seed,
                    "design": design,
                    "training_task": training_task,
                    "specification": specification,
                    "analysis_type": metadata["analysis_type"],
                    "changed_block": metadata["changed_block"],
                    "feature_columns": columns,
                    "target": target,
                    "target_column": target_column,
                })

print(f"Model jobs: {len(jobs):,}")
print(f"CatBoost fits: {len(jobs) * N_SPLITS:,}")
display(feature_specifications)


In [ ]:
COMPLETED_PATH = TABLES_DIR / "completed_jobs.csv"
OOF_PATH = TABLES_DIR / "all_oof_metrics.csv"
FOLD_PATH = TABLES_DIR / "all_fold_metrics.csv"
CONFIG_PATH = OUTPUT_DIR / "run_config.json"

input_sha256 = sha256_file(input_path)
run_config = {
    "input_file": input_path.name,
    "input_sha256": input_sha256,
    "n_rows": int(len(data)),
    "task_counts": actual_task_counts,
    "split_seeds": list(SPLIT_SEEDS),
    "n_splits": N_SPLITS,
    "catboost_random_seed": CATBOOST_RANDOM_SEED,
    "catboost_iterations": CATBOOST_ITERATIONS,
    "task_type": TASK_TYPE,
    "gpu_devices": GPU_DEVICES if TASK_TYPE.upper() == "GPU" else None,
    "cpu_thread_count": CPU_THREAD_COUNT,
    "catboost_version": catboost.__version__,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scipy_version": scipy.__version__,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "delta_mae_definition": "baseline_mae - augmented_model_mae",
    "ablation_definition": "removed_model_mae - full_all_four_mae",
    "metric_units": "milliseconds for outcomes; bits for CKIP_GPT2_tiny metrics",
}

if CONFIG_PATH.exists():
    existing_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
    protected_keys = (
        "input_sha256", "split_seeds", "n_splits", "catboost_random_seed",
        "catboost_iterations", "task_type", "gpu_devices",
    )
    mismatches = {
        key: (existing_config.get(key), run_config.get(key))
        for key in protected_keys
        if existing_config.get(key) != run_config.get(key)
    }
    if mismatches:
        raise RuntimeError(
            "The existing output directory contains a different run "
            f"configuration: {mismatches}"
        )
else:
    CONFIG_PATH.write_text(
        json.dumps(run_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

completed = (
    pd.read_csv(COMPLETED_PATH) if COMPLETED_PATH.exists() else pd.DataFrame()
)
oof_rows = pd.read_csv(OOF_PATH).to_dict("records") if OOF_PATH.exists() else []
fold_rows = (
    pd.read_csv(FOLD_PATH).to_dict("records") if FOLD_PATH.exists() else []
)

job_key_columns = [
    "split_seed", "design", "training_task", "specification", "target"
]
if completed.empty:
    completed_keys = set()
else:
    completed_keys = set(
        map(tuple, completed[job_key_columns].itertuples(index=False, name=None))
    )


def persist_progress(completed_records):
    def atomic_csv(frame, path):
        temporary = path.with_suffix(".tmp.csv")
        frame.to_csv(temporary, index=False, encoding="utf-8-sig")
        temporary.replace(path)

    completed_frame = pd.DataFrame(completed_records).drop_duplicates(
        subset=job_key_columns, keep="last"
    )
    oof_frame = pd.DataFrame(oof_rows).drop_duplicates(
        subset=job_key_columns + ["evaluation_scope"], keep="last"
    )
    fold_frame = pd.DataFrame(fold_rows).drop_duplicates(
        subset=job_key_columns + ["evaluation_scope", "fold"], keep="last"
    )

    # Write metric tables first and the completion ledger last. This makes
    # interrupted Drive writes safe to resume without skipping an incomplete job.
    atomic_csv(oof_frame, OOF_PATH)
    atomic_csv(fold_frame, FOLD_PATH)
    atomic_csv(completed_frame, COMPLETED_PATH)


def analysis_subset(job):
    if job["training_task"] == "POOLED":
        subset = data.reset_index(drop=True)
        fold_column = f"pooled_fold_seed_{job['split_seed']}"
        evaluation_scopes = {
            "POOLED": np.ones(len(subset), dtype=bool),
            "NR_within_pooled": subset["TASK"].eq("NR").to_numpy(),
            "TSR_within_pooled": subset["TASK"].eq("TSR").to_numpy(),
        }
    else:
        subset = data.loc[data["TASK"].eq(job["training_task"])].reset_index(
            drop=True
        )
        fold_column = f"task_fold_seed_{job['split_seed']}"
        evaluation_scopes = {
            job["training_task"]: np.ones(len(subset), dtype=bool)
        }
    return subset, fold_column, evaluation_scopes


def make_catboost_model():
    kwargs = {
        "iterations": CATBOOST_ITERATIONS,
        "random_seed": CATBOOST_RANDOM_SEED,
        "verbose": False,
        "allow_writing_files": False,
    }
    if TASK_TYPE.upper() == "GPU":
        kwargs.update({"task_type": "GPU", "devices": GPU_DEVICES})
    else:
        kwargs.update({"task_type": "CPU", "thread_count": CPU_THREAD_COUNT})
    return CatBoostRegressor(**kwargs)


completed_records = completed.to_dict("records") if not completed.empty else []
pending_jobs = [
    job for job in jobs
    if tuple(job[column] for column in job_key_columns) not in completed_keys
]

print(f"Already completed: {len(completed_keys):,}/{len(jobs):,}")
print(f"Pending: {len(pending_jobs):,}")

started_all = time.perf_counter()
since_save = 0
progress = tqdm(pending_jobs, unit="model job")
for job in progress:
    subset, fold_column, evaluation_scopes = analysis_subset(job)
    predictors = subset[job["feature_columns"]].to_numpy(dtype=float)
    observed = subset[job["target_column"]].to_numpy(dtype=np.float64)
    folds = subset[fold_column].to_numpy(dtype=np.int8)
    predicted = np.full(len(subset), np.nan, dtype=np.float64)
    job_started = time.perf_counter()

    for fold in range(1, N_SPLITS + 1):
        test = np.flatnonzero(folds == fold)
        train = np.flatnonzero(folds != fold)
        model = make_catboost_model()
        model.fit(predictors[train], observed[train])
        predicted[test] = np.asarray(model.predict(predictors[test]), dtype=float)
        del model
        gc.collect()

    if np.isnan(predicted).any():
        raise RuntimeError(f"Incomplete OOF predictions for {job}")

    common = {
        "split_seed": job["split_seed"],
        "design": job["design"],
        "training_task": job["training_task"],
        "specification": job["specification"],
        "analysis_type": job["analysis_type"],
        "changed_block": job["changed_block"],
        "target": job["target"],
        "n_features": len(job["feature_columns"]),
    }
    for evaluation_scope, scope_mask in evaluation_scopes.items():
        oof_rows.append({
            **common,
            "evaluation_scope": evaluation_scope,
            "n": int(scope_mask.sum()),
            **regression_metrics(observed[scope_mask], predicted[scope_mask]),
        })
        for fold in range(1, N_SPLITS + 1):
            mask = scope_mask & (folds == fold)
            fold_rows.append({
                **common,
                "evaluation_scope": evaluation_scope,
                "fold": fold,
                "n": int(mask.sum()),
                **regression_metrics(observed[mask], predicted[mask]),
            })

    completed_records.append({
        **{column: job[column] for column in job_key_columns},
        "seconds": time.perf_counter() - job_started,
    })
    completed_keys.add(tuple(job[column] for column in job_key_columns))
    since_save += 1
    progress.set_postfix(
        seed=job["split_seed"],
        scope=job["training_task"],
        target=job["target"],
    )

    if since_save >= SAVE_EVERY_N_JOBS:
        persist_progress(completed_records)
        since_save = 0

persist_progress(completed_records)
run_config["completed_jobs"] = len(completed_keys)
run_config["expected_jobs"] = len(jobs)
run_config["latest_session_seconds"] = time.perf_counter() - started_all
CONFIG_PATH.write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Completed jobs: {len(completed_keys):,}/{len(jobs):,}")
print(f"Latest session: {(time.perf_counter() - started_all) / 60:.1f} minutes")


In [ ]:
oof = pd.read_csv(OOF_PATH)
if len(completed_keys) != len(jobs):
    raise RuntimeError(
        f"Only {len(completed_keys):,}/{len(jobs):,} jobs are complete. "
        "Finish the previous cell before summarizing."
    )

merge_keys = [
    "split_seed", "design", "training_task", "evaluation_scope", "target"
]
baseline = (
    oof.loc[oof["specification"].eq("baseline_no_llm"), merge_keys + ["mae"]]
    .rename(columns={"mae": "baseline_mae"})
)

addition = oof.loc[
    oof["analysis_type"].isin(["single_metric_addition", "full_all_four"])
].merge(baseline, on=merge_keys, validate="many_to_one")
addition["delta_mae"] = addition["baseline_mae"] - addition["mae"]
addition.to_csv(
    TABLES_DIR / "metric_additions_by_seed.csv",
    index=False,
    encoding="utf-8-sig",
)

full = (
    oof.loc[oof["specification"].eq("all_four"), merge_keys + ["mae"]]
    .rename(columns={"mae": "full_all_four_mae"})
)
ablation = oof.loc[
    oof["analysis_type"].isin(
        ["leave_one_metric_out", "leave_one_control_block_out"]
    )
].merge(full, on=merge_keys, validate="many_to_one")
ablation["mae_increase_when_removed"] = (
    ablation["mae"] - ablation["full_all_four_mae"]
)
ablation.to_csv(
    TABLES_DIR / "leave_one_out_ablation_by_seed.csv",
    index=False,
    encoding="utf-8-sig",
)


def robustness_summary(frame, value_column, group_columns):
    summary = (
        frame.groupby(group_columns, dropna=False, observed=True)[value_column]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .reset_index()
    )
    summary = summary.rename(columns={
        "count": "n_split_seeds",
        "mean": f"mean_{value_column}",
        "std": f"sd_{value_column}",
        "median": f"median_{value_column}",
        "min": f"min_{value_column}",
        "max": f"max_{value_column}",
    })
    positive = (
        frame.assign(_positive=frame[value_column].gt(0).astype(int))
        .groupby(group_columns, dropna=False, observed=True)["_positive"]
        .sum()
        .reset_index(name="positive_seed_count")
    )
    summary = summary.merge(positive, on=group_columns, validate="one_to_one")
    n = summary["n_split_seeds"].astype(float)
    sem = summary[f"sd_{value_column}"] / np.sqrt(n)
    critical = student_t.ppf(0.975, np.maximum(n - 1.0, 1.0))
    margin = critical * sem
    summary["ci95_low"] = summary[f"mean_{value_column}"] - margin
    summary["ci95_high"] = summary[f"mean_{value_column}"] + margin
    summary["positive_seed_fraction"] = (
        summary["positive_seed_count"].astype(str)
        + "/"
        + summary["n_split_seeds"].astype(str)
    )
    return summary


addition_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
addition_summary = robustness_summary(
    addition, "delta_mae", addition_groups
)
addition_summary.to_csv(
    TABLES_DIR / "metric_additions_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

ablation_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
ablation_summary = robustness_summary(
    ablation, "mae_increase_when_removed", ablation_groups
)
ablation_summary.to_csv(
    TABLES_DIR / "leave_one_out_ablation_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

performance_groups = [
    "design", "training_task", "evaluation_scope", "target",
    "specification", "analysis_type", "changed_block",
]
performance_summary = (
    oof.groupby(performance_groups, dropna=False, observed=True)
    [["mae", "mse", "rmse", "r2", "pearson", "spearman"]]
    .agg(["mean", "std", "min", "max"])
)
performance_summary.columns = [
    f"{metric}_{stat}" for metric, stat in performance_summary.columns
]
performance_summary.reset_index().to_csv(
    TABLES_DIR / "all_performance_metrics_robustness.csv",
    index=False,
    encoding="utf-8-sig",
)

main_scopes = addition_summary.loc[
    addition_summary["evaluation_scope"].isin(["POOLED", "NR", "TSR"])
].copy()
display_columns = [
    "training_task", "target", "specification", "changed_block",
    "mean_delta_mae", "sd_delta_mae", "min_delta_mae", "max_delta_mae",
    "ci95_low", "ci95_high", "positive_seed_fraction",
]
display(
    main_scopes[display_columns]
    .sort_values(["training_task", "target", "changed_block"])
    .reset_index(drop=True)
)


In [ ]:
archive_base = Path("/content/MCFIX_CKIP_GPT2_tiny_JED351_matched_regression_results")
archive_path = Path(
    shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR)
)
print("Created:", archive_path)
print(f"Size: {archive_path.stat().st_size / (1024 ** 2):.2f} MB")

print("Regression archive retained for the combined package.")


## 6. Package and download final outputs


In [ ]:

# Create one compact result archive; sentence checkpoints remain on Drive.
release_dir = Path("/content/MCFIX_CKIP_GPT2_tiny_JED351_matched_results")
if release_dir.exists():
    shutil.rmtree(release_dir)
(release_dir / "extracted_metrics").mkdir(parents=True)
(release_dir / "modeling_data").mkdir(parents=True)
(release_dir / "regression").mkdir(parents=True)

for path in PIPELINE_ROOT.joinpath("extraction").glob("*"):
    if path.is_file():
        shutil.copy2(path, release_dir / "extracted_metrics" / path.name)
for path in FINAL_DATA_DIR.glob("*"):
    if path.is_file():
        shutil.copy2(path, release_dir / "modeling_data" / path.name)
for path in OUTPUT_DIR.glob("*"):
    if path.is_file():
        shutil.copy2(path, release_dir / "regression" / path.name)
if TABLES_DIR.exists():
    shutil.copytree(TABLES_DIR, release_dir / "regression" / "tables")

manifest["regression"] = {
    "reference_sample": "exact current JED351 10,089-row complete-case sample",
    "estimator": "CatBoostRegressor",
    "iterations": CATBOOST_ITERATIONS,
    "catboost_random_seed": CATBOOST_RANDOM_SEED,
    "split_seeds": list(SPLIT_SEEDS),
    "n_splits": N_SPLITS,
    "expected_jobs": 915,
    "expected_fits": 4575,
}
(release_dir / "pipeline_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

combined_archive = Path(
    shutil.make_archive(str(release_dir), "zip", root_dir=release_dir)
)
print("Combined result ZIP:", combined_archive)
print(f"ZIP size: {combined_archive.stat().st_size / (1024 ** 2):.2f} MB")

from google.colab import files
files.download(str(combined_archive))
